# US Stock Market Cap 계산 (Shares Outstanding × 월말 종가)

**로직 요약**
1. FMP API → 분기별 Shares Outstanding + 공시일(reportedDate)
2. DB `us_stock_daily_market_cap` → 티커별 **월말 close_price** 추출
3. 공시일 기준 forward-fill로 shares 매핑 (Look-ahead bias 방지)
4. 월말 종가 × shares = 시가총액 → DB 저장 (indicator = `market_cap`)

| Cell | 내용 |
|------|------|
| Cell 1 | 라이브러리 & 환경 설정 |
| Cell 2 | DB 연결 & 테이블 준비 |
| Cell 3 | DB에서 월말 종가 추출 함수 |
| Cell 4 | FMP Shares Outstanding 수집 함수 |
| Cell 5 | 단일 티커 테스트 (AAPL) |
| Cell 6 | 전체 티커 실행 & DB 저장 |
| Cell 7 | 결과 검증 |

---
## Cell 1 — 라이브러리 & 환경 설정

In [1]:
import sys
import os
import time
import requests
import pandas as pd
import pymysql
import numpy as np
from datetime import datetime, date
from typing import Optional, List, Tuple

# ── 환경별 DATA 폴더 경로 자동 감지 ──────────────────────────────────
DATA_PATHS = [
    r"C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast\DATA",
    r"C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA",
]

DATA_DIR = None
for _p in DATA_PATHS:
    if os.path.exists(_p):
        DATA_DIR = _p
        break

if DATA_DIR is None:
    raise EnvironmentError("DATA 폴더를 찾을 수 없습니다.")

PARENT_DIR = os.path.dirname(DATA_DIR)
for _path in [DATA_DIR, PARENT_DIR]:
    if _path not in sys.path:
        sys.path.insert(0, _path)

print(f"✅ DATA 폴더  : {DATA_DIR}")
print(f"✅ 상위 폴더  : {PARENT_DIR}")

from config import get_db_info, get_engine
from us_target_ticker_list_2000 import ticker_list

print(f"✅ config 임포트 성공")
print(f"✅ 티커 리스트 로드: {len(ticker_list)}개")

# ── FMP API 설정 ──────────────────────────────────────────────────────
FMP_API_KEY     = "hT0gAk87j9xZx4PlBApvBqfVL5IahvgV"
FMP_BASE_URL    = "https://financialmodelingprep.com/api/v3"

# ── Rate Limit (Starter: 250콜/분) ───────────────────────────────────
RATE_LIMIT_PER_MIN  = 240
RATE_LIMIT_INTERVAL = 60 / RATE_LIMIT_PER_MIN

# ── 테이블 & indicator 설정 ───────────────────────────────────────────
SOURCE_TABLE      = "us_stock_daily_market_cap"   # 종가 + 시가총액 저장 테이블
CLOSE_INDICATOR   = "close_price"
MKTCAP_INDICATOR  = "market_cap"

print(f"\n⚙️  종가 테이블   : {SOURCE_TABLE} / indicator = '{CLOSE_INDICATOR}'")
print(f"⚙️  저장 indicator: '{MKTCAP_INDICATOR}'")
print(f"⚙️  Rate Limit    : {RATE_LIMIT_PER_MIN}콜/분")

✅ DATA 폴더  : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA
✅ 상위 폴더  : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy
✅ config 임포트 성공
✅ 티커 리스트 로드: 2000개

⚙️  종가 테이블   : us_stock_daily_market_cap / indicator = 'close_price'
⚙️  저장 indicator: 'market_cap'
⚙️  Rate Limit    : 240콜/분


---
## Cell 2 — DB 연결 & 테이블 준비

In [2]:
def get_connection():
    db_info = get_db_info()
    return pymysql.connect(
        host=db_info['host'],
        port=int(db_info['port']),
        user=db_info['user'],
        password=db_info['password'],
        database=db_info['database'],
        charset='utf8mb4',
        autocommit=False
    )

# ── 연결 테스트 ────────────────────────────────────────────────────────
conn_test = get_connection()
print("✅ DB 연결 성공")
conn_test.close()

# ── market_cap 레코드가 없으면 동일 테이블에 추가 저장 ──────────────
# SOURCE_TABLE에 이미 UNIQUE KEY (date, ticker, indicator) 가 있으므로
# indicator = 'market_cap' 으로 INSERT IGNORE 하면 중복 없이 공존 가능
conn = get_connection()
try:
    with conn.cursor() as cur:
        # 테이블 존재 여부 확인
        cur.execute(f"SHOW TABLES LIKE '{SOURCE_TABLE}'")
        exists = cur.fetchone()
        if exists:
            print(f"✅ 테이블 '{SOURCE_TABLE}' 확인 완료")
        else:
            print(f"❌ 테이블 '{SOURCE_TABLE}' 없음 — us_market_cap_collection 노트북 먼저 실행 필요")
        
        # indicator 종류 확인
        cur.execute(
            f"SELECT DISTINCT indicator FROM `{SOURCE_TABLE}` LIMIT 20"
        )
        indicators = [r[0] for r in cur.fetchall()]
        print(f"\n📋 현재 저장된 indicator 목록: {indicators}")
        
        # 종가 데이터 건수
        cur.execute(
            f"SELECT COUNT(DISTINCT ticker) FROM `{SOURCE_TABLE}` "
            f"WHERE indicator = %s", (CLOSE_INDICATOR,)
        )
        close_ticker_cnt = cur.fetchone()[0]
        print(f"📋 종가(close_price) 보유 티커 수: {close_ticker_cnt:,}개")
        
        # 기존 market_cap 건수
        cur.execute(
            f"SELECT COUNT(*) FROM `{SOURCE_TABLE}` "
            f"WHERE indicator = %s", (MKTCAP_INDICATOR,)
        )
        mktcap_cnt = cur.fetchone()[0]
        print(f"📋 기존 market_cap 레코드 수: {mktcap_cnt:,}건")
finally:
    conn.close()

✅ DB 연결 성공
✅ 테이블 'us_stock_daily_market_cap' 확인 완료

📋 현재 저장된 indicator 목록: ['close_price', 'market_cap']
📋 종가(close_price) 보유 티커 수: 2,000개
📋 기존 market_cap 레코드 수: 261,144건


---
## Cell 3 — DB에서 월말 종가 추출 함수

DB의 `close_price`에서 **각 월의 마지막 날짜 종가**만 추출합니다.

In [3]:
def get_monthly_close_from_db(ticker: str, from_date: Optional[str] = None) -> pd.DataFrame:
    """
    DB에서 특정 티커의 월말 종가를 추출.
    
    - from_date: 'YYYY-MM-DD' 이후 데이터만 조회 (None이면 전체)
    - 각 월의 마지막 거래일 종가를 month_end_date로 반환
    
    Returns
    -------
    DataFrame: [month_end_date, year_month, close_price]
    """
    conn = get_connection()
    try:
        with conn.cursor() as cur:
            if from_date:
                cur.execute(
                    f"""
                    SELECT date, value
                    FROM `{SOURCE_TABLE}`
                    WHERE ticker    = %s
                      AND indicator = %s
                      AND date      >= %s
                    ORDER BY date ASC
                    """,
                    (ticker, CLOSE_INDICATOR, from_date)
                )
            else:
                cur.execute(
                    f"""
                    SELECT date, value
                    FROM `{SOURCE_TABLE}`
                    WHERE ticker    = %s
                      AND indicator = %s
                    ORDER BY date ASC
                    """,
                    (ticker, CLOSE_INDICATOR)
                )
            rows = cur.fetchall()
    finally:
        conn.close()
    
    if not rows:
        return pd.DataFrame()
    
    df = pd.DataFrame(rows, columns=['date', 'close_price'])
    df['date'] = pd.to_datetime(df['date'])
    df['close_price'] = pd.to_numeric(df['close_price'], errors='coerce')
    
    # ── 월별 마지막 거래일만 추출 ─────────────────────────────────────
    df['year_month'] = df['date'].dt.to_period('M')
    df_monthly = (
        df.sort_values('date')
          .groupby('year_month', as_index=False)
          .last()   # 각 월의 마지막 날짜
          .rename(columns={'date': 'month_end_date'})
    )
    
    return df_monthly[['month_end_date', 'year_month', 'close_price']]


def get_last_mktcap_date_in_db(ticker: str) -> Optional[str]:
    """DB에서 해당 티커의 market_cap 마지막 저장일 반환"""
    conn = get_connection()
    try:
        with conn.cursor() as cur:
            cur.execute(
                f"SELECT MAX(date) FROM `{SOURCE_TABLE}` "
                f"WHERE ticker = %s AND indicator = %s",
                (ticker, MKTCAP_INDICATOR)
            )
            row = cur.fetchone()
            last_date = row[0] if row and row[0] else None
    finally:
        conn.close()
    return str(last_date) if last_date else None


# ── 테스트 ────────────────────────────────────────────────────────────
print("🧪 AAPL 월말 종가 추출 테스트")
df_close_test = get_monthly_close_from_db('AAPL', from_date='2020-01-01')
if df_close_test.empty:
    print("❌ 데이터 없음 — DB에 AAPL close_price 확인 필요")
else:
    print(f"✅ {len(df_close_test)}개월치 월말 종가 추출")
    print(f"   기간: {df_close_test['month_end_date'].min().date()} ~ {df_close_test['month_end_date'].max().date()}")
    print()
    print(df_close_test.tail(8).to_string(index=False))

🧪 AAPL 월말 종가 추출 테스트
✅ 77개월치 월말 종가 추출
   기간: 2020-01-31 ~ 2026-05-06

month_end_date year_month  close_price
    2025-10-31    2025-10       270.11
    2025-11-28    2025-11       278.85
    2025-12-31    2025-12       271.86
    2026-01-30    2026-01       259.24
    2026-02-27    2026-02       264.18
    2026-03-31    2026-03       253.79
    2026-04-30    2026-04       271.35
    2026-05-06    2026-05       287.51


---
## Cell 4 — FMP Shares Outstanding 수집 함수

FMP Income Statement에서 `weightedAverageShsOut` + `date`(분기말) + `acceptedDate`(공시일)를 수집합니다.

**Look-ahead 방지 매핑 원칙**
```
acceptedDate(공시일) 이후의 월에 해당 shares를 적용
```

In [4]:
def fetch_shares_outstanding_fmp(ticker: str) -> pd.DataFrame:
    """
    FMP quarterly income statement에서 shares outstanding + 공시일 수집.
    
    Returns
    -------
    DataFrame: [period, accepted_date, shares]
        - period       : 분기말 날짜 (재무제표 기준)
        - accepted_date: SEC 공시일 (이 날짜 이후부터 해당 shares 사용 가능)
        - shares       : weightedAverageShsOut
    """
    url = f"{FMP_BASE_URL}/income-statement/{ticker}"
    params = {
        "apikey" : FMP_API_KEY,
        "period" : "quarter",
        "limit"  : 60,          # 최대 15년치 분기 데이터
    }
    
    try:
        resp = requests.get(url, params=params, timeout=15)
        resp.raise_for_status()
        data = resp.json()
    except requests.exceptions.RequestException as e:
        print(f"  ⚠️  API 오류 [{ticker}]: {e}")
        return pd.DataFrame()
    
    if not data or not isinstance(data, list):
        return pd.DataFrame()
    
    df = pd.DataFrame(data)
    
    # 필요 컬럼 확인
    needed = ['date', 'acceptedDate', 'weightedAverageShsOut']
    missing = [c for c in needed if c not in df.columns]
    if missing:
        print(f"  ⚠️  컬럼 없음 [{ticker}]: {missing} | 실제 컬럼: {df.columns.tolist()}")
        return pd.DataFrame()
    
    df = df[needed].copy()
    df.columns = ['period', 'accepted_date', 'shares']
    
    df['period']        = pd.to_datetime(df['period'])
    df['accepted_date'] = pd.to_datetime(df['accepted_date'])
    df['shares']        = pd.to_numeric(df['shares'], errors='coerce')
    
    # 유효한 데이터만
    df = df.dropna(subset=['accepted_date', 'shares'])
    df = df[df['shares'] > 0]
    df = df.sort_values('accepted_date').reset_index(drop=True)
    
    return df


def map_shares_to_months(df_shares: pd.DataFrame, df_close: pd.DataFrame) -> pd.DataFrame:
    """
    공시일(accepted_date) 기준으로 월별 종가에 shares를 매핑.
    
    원칙: 해당 월이 accepted_date 이후인 가장 최근 공시의 shares 사용
    → Look-ahead bias 완전 방지
    
    Returns
    -------
    DataFrame: [month_end_date, close_price, shares, market_cap]
    """
    if df_shares.empty or df_close.empty:
        return pd.DataFrame()
    
    result_rows = []
    
    for _, row in df_close.iterrows():
        month_end = row['month_end_date']   # 해당 월 마지막 거래일
        close     = row['close_price']
        
        # 공시일이 월말 이하인 것 중 가장 최근 공시 선택 (look-ahead 방지)
        valid_shares = df_shares[df_shares['accepted_date'] <= month_end]
        
        if valid_shares.empty:
            continue   # 해당 월 이전에 공시된 shares 없으면 스킵
        
        latest = valid_shares.iloc[-1]   # 가장 최근 공시
        shares = latest['shares']
        
        market_cap = close * shares
        
        result_rows.append({
            'month_end_date' : month_end,
            'close_price'    : close,
            'shares'         : shares,
            'accepted_date'  : latest['accepted_date'],
            'period'         : latest['period'],
            'market_cap'     : market_cap,
        })
    
    if not result_rows:
        return pd.DataFrame()
    
    return pd.DataFrame(result_rows)


def upsert_market_cap_to_db(conn, ticker: str, df_calc: pd.DataFrame) -> int:
    """
    계산된 시가총액을 DB에 upsert.
    INSERT IGNORE → 중복(date+ticker+indicator) 시 기존 데이터 유지
    """
    if df_calc.empty:
        return 0
    
    SQL = f"""
        INSERT IGNORE INTO `{SOURCE_TABLE}`
            (date, ticker, indicator, value)
        VALUES (%s, %s, %s, %s)
    """
    rows = [
        (
            str(r['month_end_date'].date()) if hasattr(r['month_end_date'], 'date') else str(r['month_end_date']),
            ticker,
            MKTCAP_INDICATOR,
            float(r['market_cap'])
        )
        for _, r in df_calc.iterrows()
    ]
    
    with conn.cursor() as cur:
        cur.executemany(SQL, rows)
    conn.commit()
    return len(rows)


print("✅ 함수 정의 완료")
print("   - fetch_shares_outstanding_fmp()")
print("   - map_shares_to_months()")
print("   - upsert_market_cap_to_db()")

✅ 함수 정의 완료
   - fetch_shares_outstanding_fmp()
   - map_shares_to_months()
   - upsert_market_cap_to_db()


---
## Cell 5 — 단일 티커 테스트 (AAPL)

전체 실행 전 데이터 품질 및 계산 결과를 육안으로 확인합니다.

In [5]:
TEST_TICKER   = "STRL"
TEST_FROM_DATE = "2015-01-01"   # 종가 조회 시작일

print(f"{'='*60}")
print(f"🧪 단일 티커 테스트: {TEST_TICKER}")
print(f"{'='*60}")

# ── Step 1: DB에서 월말 종가 추출 ────────────────────────────────────
print("\n[Step 1] DB 월말 종가 추출")
df_close = get_monthly_close_from_db(TEST_TICKER, from_date=TEST_FROM_DATE)
if df_close.empty:
    print("  ❌ 종가 데이터 없음")
else:
    print(f"  ✅ {len(df_close)}개월치 | 기간: {df_close['month_end_date'].min().date()} ~ {df_close['month_end_date'].max().date()}")
    print(df_close.tail(4).to_string(index=False))

# ── Step 2: FMP Shares Outstanding 수집 ─────────────────────────────
print("\n[Step 2] FMP Shares Outstanding 수집")
df_shares = fetch_shares_outstanding_fmp(TEST_TICKER)
if df_shares.empty:
    print("  ❌ Shares 데이터 없음")
else:
    print(f"  ✅ {len(df_shares)}분기 | 기간: {df_shares['period'].min().date()} ~ {df_shares['period'].max().date()}")
    print(df_shares.tail(6).to_string(index=False))

# ── Step 3: 매핑 & 시가총액 계산 ─────────────────────────────────────
print("\n[Step 3] Look-ahead 방지 매핑 & 시가총액 계산")
df_calc = map_shares_to_months(df_shares, df_close)
if df_calc.empty:
    print("  ❌ 계산 실패")
else:
    print(f"  ✅ {len(df_calc)}개월치 시가총액 계산 완료")
    df_display = df_calc[['month_end_date','close_price','shares','accepted_date','market_cap']].tail(8).copy()
    df_display['market_cap_조$'] = (df_display['market_cap'] / 1e12).round(2).astype(str) + '조$'
    df_display['shares_억'] = (df_display['shares'] / 1e8).round(2).astype(str) + '억주'
    print(df_display[['month_end_date','close_price','shares_억','accepted_date','market_cap_조$']].to_string(index=False))

# ── Step 4: DB 저장 미리보기 (실제 저장 X) ───────────────────────────
print("\n[Step 4] DB 저장 예정 데이터 형태")
if not df_calc.empty:
    preview = df_calc[['month_end_date', 'market_cap']].copy()
    preview.insert(1, 'ticker', TEST_TICKER)
    preview.insert(2, 'indicator', MKTCAP_INDICATOR)
    preview.columns = ['date', 'ticker', 'indicator', 'value']
    print(preview.tail(5).to_string(index=False))
    print(f"\n  → 저장 예정: {len(preview)}건 (INSERT IGNORE)")

🧪 단일 티커 테스트: STRL

[Step 1] DB 월말 종가 추출
  ✅ 137개월치 | 기간: 2015-01-30 ~ 2026-05-06
month_end_date year_month  close_price
    2026-02-27    2026-02       428.13
    2026-03-31    2026-03       407.27
    2026-04-30    2026-04       515.62
    2026-05-06    2026-05       886.22

[Step 2] FMP Shares Outstanding 수집
  ✅ 60분기 | 기간: 2011-06-30 ~ 2026-03-31
    period       accepted_date   shares
2024-12-31 2025-02-26 09:10:57 30696000
2025-03-31 2025-05-06 09:10:22 30547000
2025-06-30 2025-08-05 09:10:46 30408000
2025-09-30 2025-11-04 09:10:28 31006060
2025-12-31 2026-02-26 09:11:42 30696000
2026-03-31 2026-05-05 09:10:55 30652000

[Step 3] Look-ahead 방지 매핑 & 시가총액 계산
  ✅ 137개월치 시가총액 계산 완료
month_end_date  close_price shares_억       accepted_date market_cap_조$
    2025-10-31       377.90    0.3억주 2025-08-05 09:10:46        0.01조$
    2025-11-28       344.31   0.31억주 2025-11-04 09:10:28        0.01조$
    2025-12-31       306.23   0.31억주 2025-11-04 09:10:28        0.01조$
    2026-01-30       357.9

---
## Cell 6 — 전체 티커 실행 & DB 저장

| 파라미터 | 설명 | 기본값 |
|---------|------|-------|
| `CLOSE_FROM_DATE` | 종가 조회 시작일 | `'2010-01-01'` |
| `TEST_MODE` | True면 앞 5개만 실행 | `True` |

In [ ]:
# ── 실행 파라미터 ─────────────────────────────────────────────────────
CLOSE_FROM_DATE = "2026-04-15"   # 종가 조회 시작일
TEST_MODE       = False           # ← 전체 실행 시 False

# ─────────────────────────────────────────────────────────────────────
target_tickers = ticker_list[:5] if TEST_MODE else ticker_list
total = len(target_tickers)

print(f"{'🧪 테스트 모드' if TEST_MODE else '🚀 전체 실행'}: {total}개 티커")
print(f"📅 종가 조회 시작일: {CLOSE_FROM_DATE}")
print("-" * 65)

success_count  = 0
skip_count     = 0
error_count    = 0
total_inserted = 0
error_tickers  = []
last_call_time = 0.0

conn = get_connection()

try:
    for idx, ticker in enumerate(target_tickers, start=1):
        
        pct = idx / total * 100
        print(f"[{idx:4d}/{total}] ({pct:5.1f}%) {ticker:<8}", end="  ")
        
        try:
            # ── Rate Limit ────────────────────────────────────────────
            elapsed = time.time() - last_call_time
            if elapsed < RATE_LIMIT_INTERVAL:
                time.sleep(RATE_LIMIT_INTERVAL - elapsed)
            
            # ── 증분: 이미 저장된 market_cap 이후만 계산 ──────────────
            last_mktcap_date = get_last_mktcap_date_in_db(ticker)
            close_from = last_mktcap_date if last_mktcap_date else CLOSE_FROM_DATE
            
            # ── DB에서 월말 종가 추출 ──────────────────────────────────
            df_close = get_monthly_close_from_db(ticker, from_date=close_from)
            if df_close.empty:
                print(f"→ 종가 없음 (skip)")
                skip_count += 1
                continue
            
            # ── FMP Shares Outstanding 수집 ───────────────────────────
            last_call_time = time.time()
            df_shares = fetch_shares_outstanding_fmp(ticker)
            if df_shares.empty:
                print(f"→ Shares 없음 (skip)")
                skip_count += 1
                continue
            
            # ── 시가총액 계산 ──────────────────────────────────────────
            df_calc = map_shares_to_months(df_shares, df_close)
            if df_calc.empty:
                print(f"→ 매핑 실패 (skip)")
                skip_count += 1
                continue
            
            # ── DB 저장 ────────────────────────────────────────────────
            inserted = upsert_market_cap_to_db(conn, ticker, df_calc)
            total_inserted += inserted
            
            cal_range = f"{df_calc['month_end_date'].min().date()}~{df_calc['month_end_date'].max().date()}"
            print(f"→ {len(df_calc):3d}개월 계산  💾 DB저장 {inserted}건  [{cal_range}]")
            success_count += 1
        
        except Exception as e:
            print(f"→ ❌ 오류: {e}")
            error_count += 1
            error_tickers.append(ticker)
            continue

except Exception as e:
    print(f"\n❌ 전체 예외 발생: {e}")
    import traceback
    traceback.print_exc()
finally:
    conn.close()

print("\n" + "=" * 65)
print("📊 실행 완료 요약")
print(f"   성공   : {success_count:,}개 티커")
print(f"   스킵   : {skip_count:,}개 티커")
print(f"   오류   : {error_count:,}개 티커")
print(f"   DB 저장: {total_inserted:,}건")
if error_tickers:
    print(f"   오류 티커: {error_tickers[:10]}{'...' if len(error_tickers)>10 else ''}")
print("=" * 65)

🚀 전체 실행: 2000개 티커
📅 종가 조회 시작일: 2026-04-15
-----------------------------------------------------------------
[   1/2000] (  0.1%) NVDA      →   2개월 계산  💾 DB저장 2건  [2026-04-30~2026-05-06]
[   2/2000] (  0.1%) GOOG      →   2개월 계산  💾 DB저장 2건  [2026-04-30~2026-05-06]
[   3/2000] (  0.1%) AAPL      →   2개월 계산  💾 DB저장 2건  [2026-04-30~2026-05-06]
[   4/2000] (  0.2%) MSFT      →   2개월 계산  💾 DB저장 2건  [2026-04-30~2026-05-06]
[   5/2000] (  0.2%) AMZN      →   2개월 계산  💾 DB저장 2건  [2026-04-30~2026-05-06]
[   6/2000] (  0.3%) TSM       →   1개월 계산  💾 DB저장 1건  [2026-04-13~2026-04-13]
[   7/2000] (  0.4%) META      →   2개월 계산  💾 DB저장 2건  [2026-04-30~2026-05-06]
[   8/2000] (  0.4%) AVGO      →   2개월 계산  💾 DB저장 2건  [2026-04-30~2026-05-06]
[   9/2000] (  0.4%) TSLA      →   2개월 계산  💾 DB저장 2건  [2026-04-30~2026-05-06]
[  10/2000] (  0.5%) LLY       →   2개월 계산  💾 DB저장 2건  [2026-04-30~2026-05-06]
[  11/2000] (  0.5%) WMT       →   2개월 계산  💾 DB저장 2건  [2026-04-30~2026-05-06]
[  12/2000] (  0.6%) XOM       →  

---
## Cell 7 — 결과 검증

In [7]:
conn = get_connection()
try:
    with conn.cursor() as cur:
        
        # 1) indicator별 레코드 수
        cur.execute(
            f"SELECT indicator, COUNT(*), COUNT(DISTINCT ticker), MIN(date), MAX(date) "
            f"FROM `{SOURCE_TABLE}` GROUP BY indicator"
        )
        rows = cur.fetchall()
        df_summary = pd.DataFrame(rows, columns=['indicator','레코드수','티커수','최초일','최근일'])
        print("📊 테이블 요약")
        print(df_summary.to_string(index=False))
        
        print()
        
        # 2) AAPL 시가총액 최근 12개월 샘플
        cur.execute(
            f"""
            SELECT m.date, m.value AS market_cap, c.value AS close_price,
                   ROUND(m.value / 1e12, 2) AS market_cap_조달러
            FROM `{SOURCE_TABLE}` m
            JOIN `{SOURCE_TABLE}` c
              ON m.date = c.date AND m.ticker = c.ticker
             AND c.indicator = '{CLOSE_INDICATOR}'
            WHERE m.ticker    = 'AAPL'
              AND m.indicator = '{MKTCAP_INDICATOR}'
            ORDER BY m.date DESC
            LIMIT 12
            """
        )
        rows2 = cur.fetchall()
        df_aapl = pd.DataFrame(rows2, columns=['date','market_cap','close_price','시가총액(조$)'])
        print("🔍 AAPL 최근 12개월 시가총액 검증 (종가와 함께)")
        print(df_aapl.to_string(index=False))
        
        print()
        
        # 3) 시가총액 상위 10개 티커 (최근월 기준)
        cur.execute(
            f"""
            SELECT ticker, value, ROUND(value/1e12,2) AS 조달러
            FROM `{SOURCE_TABLE}`
            WHERE indicator = '{MKTCAP_INDICATOR}'
              AND date = (
                  SELECT MAX(date) FROM `{SOURCE_TABLE}` WHERE indicator = '{MKTCAP_INDICATOR}'
              )
            ORDER BY value DESC
            LIMIT 10
            """
        )
        rows3 = cur.fetchall()
        df_top = pd.DataFrame(rows3, columns=['ticker','시가총액','시가총액(조$)'])
        print("🏆 최근월 시가총액 상위 10 티커")
        print(df_top.to_string(index=False))

finally:
    conn.close()

📊 테이블 요약
  indicator    레코드수  티커수        최초일        최근일
close_price 5416233 2000 2015-01-02 2026-04-15
 market_cap  259369 1993 2015-01-30 2026-04-15

🔍 AAPL 최근 12개월 시가총액 검증 (종가와 함께)
      date   market_cap  close_price  시가총액(조$)
2026-04-15 3.929352e+12   266.430000      3.93
2026-04-06 3.817708e+12   258.859985      3.82
2026-04-02 3.774349e+12   255.920000      3.77
2026-03-31 3.742935e+12   253.790000      3.74
2026-02-27 3.896168e+12   264.180000      3.90
2026-01-30 3.826852e+12   259.240000      3.83
2025-12-31 4.009434e+12   271.860000      4.01
2025-11-28 4.112524e+12   278.850000      4.11
2025-10-31 3.987459e+12   270.110000      3.99
2025-09-30 3.755323e+12   254.380000      3.76
2025-08-29 3.470145e+12   231.910000      3.47
2025-07-31 3.102860e+12   207.140000      3.10

🏆 최근월 시가총액 상위 10 티커
ticker         시가총액  시가총액(조$)
  NVDA 4.833336e+12      4.83
  GOOG 4.038056e+12      4.04
  AAPL 3.929352e+12      3.93
  MSFT 3.055776e+12      3.06
  AMZN 2.661186e+12      2.66
  AVG